# Hopf-GAE: Normative Graph Autoencoder for Brain Dynamics Anomaly Detection

## Physics-Informed Graph Neural Network with Coupled Edge-Node Bottleneck

---

**Kernel:** Python 3.10+ | **Dependencies:** PyTorch, PyTorch Geometric, rdata, nibabel

---

### Architecture

| Component | Detail | Parameters |
|-----------|--------|------------|
| Encoder (frozen) | Multi-relational GAT, learned relation weights, edge-attribute attention, residual | 5,485 |
| Bottleneck | h (32) → Linear → z (6), deterministic | 198 |
| Node decoder | Dropout(z) → Linear → 7 features | 49 |
| Edge decoders | MLP([z_i ∥ z_j]) → P(edge) × 3 relations, **coupled through z** | 339 |
| **Trainable** | | **586** |

### Pipeline

| Section | Content |
|---------|---------|
| S1–S6 | Libraries, data loading, ROI metadata, SC matrix, simulator, QC |
| S7 | HopfEncoder + synthetic pre-training |
| S8–S9 | HC data + MDD graphs + derived feature augmentation |
| S10 | HopfGAE model definition (coupled edge-node bottleneck) |
| S11 | GAE training on HC data |
| S12 | Anomaly scoring (physics-only, label-free) |
| S13 | Statistical analysis (HC–MDD, intervention, enrichment, heterogeneity) |
| F0–F15 | Report figures |
| S14 | Summary statistics export |

# 1 — Libraries and Configuration

In [1]:
# =============================================================================
# S1 --- LIBRARIES AND CONFIGURATION
# =============================================================================

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats as sp_stats
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import global_mean_pool

from config import *
from models import *
from utils import *

# ── v6 configuration ───────────────────────────────────────────────────
LATENT_DIM = 6          # wider bottleneck for joint dynamics + connectivity encoding
NOISE_SIGMA = 0.1       # denoising noise level
Z_DROPOUT = 0.3         # dropout on z before node decoder
LAMBDA_GRAPH = 0.1      # graph-level loss weight
N_FEATURES_OUT = 7      # training reconstruction targets

# Scoring: physics-only (a, omega, chisq)
SCORE_WEIGHTS = torch.tensor([2.0, 1.0, 1.0])

# Training: all 7 features shape the latent space
TRAIN_WEIGHTS = torch.tensor([2.0, 1.0, 1.0, 0.5, 0.5, 0.5, 0.5])

HC_HOLDOUT_FRAC = 0.15
OUTLIER_N_SD = 5
N_PERMUTATIONS = 10000
N_PERMS = 10000
N_BOOT = 10000
EXCLUDED_SESSIONS = {("LS_E4209", "rest2")}

seed_everything(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_V6 = RESULTS_DIR / "v6"
RESULTS_V6.mkdir(parents=True, exist_ok=True)

log.info('Configuration loaded. LATENT_DIM=%d, λ_edge=%.2f, scoring=physics-only',
         LATENT_DIM, LAMBDA_EDGE)

[20:05:10] Configuration loaded. LATENT_DIM=6, λ_edge=0.50, scoring=physics-only


# 2 — Data Loading

In [2]:
# =============================================================================
# S2 --- DATA LOADING
# =============================================================================

all_data = load_all_data()
ukf_df = all_data["ukf_df"]
plv_all = all_data["plv_all"]
mvar_all = all_data["mvar_all"]
group_df = all_data["group_df"]

[20:05:10] UKF: 8205 rows, 19 subjects
[20:05:10] PLV: 38 matrices
[20:05:10] MVAR: 38 matrices


# 3 — ROI Metadata and Yeo Network Assignment

In [3]:
# =============================================================================
# S3 --- ROI METADATA
# =============================================================================

roi_meta, network_assignment, N_NETWORKS = build_roi_meta_and_assignment(ukf_df)

circuit_mask = roi_meta['is_depression_circuit'].values
limbic_mask = (roi_meta['network'] == 'Limbic').values
subcort_mask = (roi_meta['network'] == 'Subcortical').values
lh_mask = roi_meta['roi_name'].str.contains('_LH_|lh|-lh', case=False, regex=True).values
rh_mask = roi_meta['roi_name'].str.contains('_RH_|rh|-rh', case=False, regex=True).values
amyg_mask = roi_meta['roi_name'].str.contains('Amyg', case=False).values

[20:05:10] ROI metadata: 216 ROIs, 8 networks, 69 circuit ROIs


# 4 — Structural Connectivity Matrix

SC = exp(−dist/40mm) from atlas centroids, row-normalised.

In [4]:
# =============================================================================
# S4 --- STRUCTURAL CONNECTIVITY
# =============================================================================

sc_matrix, centroids = load_or_build_sc(roi_meta)
log.info('SC matrix: shape %s, density %.2f%%',
         sc_matrix.shape, 100 * (sc_matrix > 0).sum() / sc_matrix.size)

[20:05:11] SC from atlas centroids: (216, 216)
[20:05:11] SC matrix: shape (216, 216), density 99.54%


# 5 — Stuart-Landau Simulator

In [5]:
# =============================================================================
# S5 --- SIMULATOR
# =============================================================================

simulator = StuartLandauSimulator(sc_matrix, n_rois=N_ROIS_216, TR=TR, n_TRs=N_VOLS)
log.info('Simulator ready: %d ROIs, %d TRs', simulator.n_rois, simulator.n_TRs)

[20:05:11] Simulator ready: 216 ROIs, 260 TRs


# 6 — Data Inspection and Quality Control

In [6]:
# =============================================================================
# S6 --- QC
# =============================================================================

log.info('=' * 70)
log.info('DATA QUALITY CONTROL')
log.info('=' * 70)

for sess in sorted(ukf_df['session'].unique()):
    a_vals = ukf_df.loc[ukf_df['session'] == sess, 'a']
    log.info('  %s: n=%d, mean_a=%.4f +/- %.4f, subcritical=%.1f%%',
             sess, len(a_vals), a_vals.mean(), a_vals.std(), (a_vals < 0).mean() * 100)

test_subj = ukf_df['subject'].unique()[0]
test_sess = ukf_df['session'].unique()[0]
test_graph = build_subject_graph(
    test_subj, test_sess, ukf_df, roi_meta,
    plv_mat=np.array(plv_all[f"{test_subj}|{test_sess}"]) if plv_all else None,
    mvar_mat=np.array(mvar_all[f"{test_subj}|{test_sess}"]) if mvar_all else None,
    sc_mat=sc_matrix,
    group=ukf_df[ukf_df['subject'] == test_subj]['group'].iloc[0],
)
log.info('Test graph: %s, x=%s, a=[%.4f, %.4f]',
         test_subj, test_graph.x.shape,
         test_graph.x[:, 0].min().item(), test_graph.x[:, 0].max().item())
for rel in EDGE_RELATIONS:
    ei = f'edge_index_{rel}'
    if hasattr(test_graph, ei):
        log.info('  %s edges: %d', rel.upper(), getattr(test_graph, ei).size(1))

[20:05:11] ======================================================================
[20:05:11] DATA QUALITY CONTROL
[20:05:11] ======================================================================
[20:05:11]   rest1: n=4101, mean_a=-0.2854 +/- 0.1573, subcritical=100.0%
[20:05:11]   rest2: n=4104, mean_a=-0.2906 +/- 0.1620, subcritical=100.0%
[20:05:11] Test graph: AC_E5694, x=torch.Size([216, 11]), a=[-0.7213, -0.0542]
[20:05:11]   PLV edges: 4644
[20:05:11]   MVAR edges: 2337
[20:05:11]   SC edges: 7310


# 7 — HopfEncoder + Synthetic Pre-Training

The **HopfEncoder** produces 32-dim per-node embeddings via two multi-relational
GAT layers with learned relation weights, edge-attribute attention, and a residual
input projection. Pre-trained on 200 synthetic Stuart-Landau graphs to recover
the bifurcation parameter $a_j$ per node.

In [7]:
# =============================================================================
# S7 --- ENCODER + PRE-TRAINING
# =============================================================================

log.info('=' * 70)
log.info('STAGE 1: ENCODER + SYNTHETIC PRE-TRAINING')
log.info('=' * 70)

encoder = HopfEncoder(
    n_node_features=3 + N_NETWORKS, hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
)
log.info('HopfEncoder: %d total parameters', sum(p.numel() for p in encoder.parameters()))
for name, module in encoder.named_children():
    n = sum(p.numel() for p in module.parameters())
    log.info('  %-20s %6d params', name, n)

# ── Synthetic data generation ──
seed_everything(SEED)

syn_train, syn_val = [], []
for i in range(N_SYN + N_VAL_SYN):
    result = simulator.generate_graph(
        a_mean=np.random.uniform(-0.40, -0.05),
        a_std=np.random.uniform(0.05, 0.20),
        G=np.random.uniform(0.2, 1.0), seed=SEED + i,
    )
    nr = simulator.n_rois
    x = torch.zeros(nr, 3 + N_NETWORKS, dtype=torch.float32)
    x[:, 0] = torch.tensor(result['a_true'], dtype=torch.float32)
    x[:, 1] = torch.tensor(result['omega'] / (2 * np.pi * TR), dtype=torch.float32)
    x[:, 2] = 0.5
    if network_assignment is not None:
        for j in range(min(nr, len(network_assignment))):
            x[j, 3 + network_assignment[j].item()] = 1.0

    ei_plv, ea_plv = matrix_to_edge_index(result['plv'][:nr, :nr], directed=False, top_k_pct=PLV_TOP_K)
    ei_sc, ea_sc = matrix_to_edge_index(sc_matrix[:nr, :nr], directed=False, top_k_pct=SC_TOP_K)
    ei_mvar, ea_mvar = matrix_to_edge_index(result['mvar'][:nr, :nr], directed=True, threshold=1e-10)

    data = Data(
        x=x, num_nodes=nr,
        edge_index_plv=ei_plv, edge_attr_plv=ea_plv,
        edge_index_mvar=ei_mvar, edge_attr_mvar=ea_mvar,
        edge_index_sc=ei_sc, edge_attr_sc=ea_sc,
        a_true=torch.tensor(result['a_true'][:nr], dtype=torch.float32),
    )
    (syn_train if i < N_SYN else syn_val).append(data)
    if (i + 1) % 50 == 0:
        log.info('  Generated %d / %d', i + 1, N_SYN + N_VAL_SYN)

log.info('Synthetic dataset: %d train + %d val', len(syn_train), len(syn_val))

# ── Pre-training loop ──
encoder.train()
pt_criterion = HopfPhysicsLoss(lambda_physics=LAMBDA_PHYSICS, lambda_subcrit=LAMBDA_SUBCRIT)
pt_optimizer = torch.optim.AdamW(encoder.parameters(), lr=PRETRAIN_LR, weight_decay=PRETRAIN_WD)
pt_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(pt_optimizer, T_max=PRETRAIN_EPOCHS)

pt_train_loader = PyGDataLoader(syn_train, batch_size=PRETRAIN_BS, shuffle=True)
pt_val_loader = PyGDataLoader(syn_val, batch_size=max(1, len(syn_val)))
pretrain_history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'r2': [], 'r2_graph': []}

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    ep_loss, nb = 0.0, 0
    for batch in pt_train_loader:
        pt_optimizer.zero_grad()
        out = encoder(batch)
        loss, _ = pt_criterion(out['a_pred'], batch.a_true)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        pt_optimizer.step()
        ep_loss += loss.item(); nb += 1
    pt_scheduler.step()

    encoder.eval()
    with torch.no_grad():
        all_preds, all_trues, val_loss = [], [], 0.0
        for batch in pt_val_loader:
            out = encoder(batch)
            vl, _ = pt_criterion(out['a_pred'], batch.a_true)
            val_loss += vl.item()
            all_preds.append(out['a_pred'].squeeze().cpu().numpy())
            all_trues.append(batch.a_true.cpu().numpy())

    vp, vt = np.concatenate(all_preds), np.concatenate(all_trues)
    ss_res = np.sum((vp - vt) ** 2)
    ss_tot = np.sum((vt - vt.mean()) ** 2)
    r2 = 1 - ss_res / max(ss_tot, 1e-12)

    # Graph-mean R²
    bid = batch.batch.cpu().numpy() if hasattr(batch, 'batch') and batch.batch is not None else np.zeros(len(vt), dtype=int)
    ugs = np.unique(bid)
    gp = np.array([vp[bid == g].mean() for g in ugs])
    gt = np.array([vt[bid == g].mean() for g in ugs])
    r2g = 1 - np.sum((gp - gt) ** 2) / max(np.sum((gt - gt.mean()) ** 2), 1e-12)

    pretrain_history['epoch'].append(epoch)
    pretrain_history['train_loss'].append(ep_loss / max(nb, 1))
    pretrain_history['val_loss'].append(val_loss)
    pretrain_history['r2'].append(r2)
    pretrain_history['r2_graph'].append(r2g)

    if epoch % 20 == 0 or epoch == 1:
        log.info('  Epoch %3d | Train %.4f | Val %.4f | R2_roi %.3f | R2_graph %.3f | LR %.1e',
                 epoch, ep_loss / max(nb, 1), val_loss, r2, r2g,
                 pt_scheduler.get_last_lr()[0])

log.info('Pre-training complete. R2_roi = %.3f, R2_graph = %.3f', r2, r2g)
for lname in ['conv1', 'conv2']:
    conv = getattr(encoder, lname)
    wts = {k: f'{w:.3f}' for k, w in zip(RELATION_KEYS, conv.relation_weights.tolist())}
    log.info('Learned relation weights (%s): %s', lname, wts)

[20:05:11] ======================================================================
[20:05:11] STAGE 1: ENCODER + SYNTHETIC PRE-TRAINING
[20:05:11] ======================================================================
[20:05:11] HopfEncoder: 5485 total parameters
[20:05:11]   conv1                  1286 params
[20:05:11]   conv2                  3302 params
[20:05:11]   input_proj              352 params
[20:05:11]   physics_head            545 params
[20:05:44]   Generated 50 / 220
[20:06:18]   Generated 100 / 220
[20:06:50]   Generated 150 / 220
[20:07:24]   Generated 200 / 220
[20:07:37] Synthetic dataset: 200 train + 20 val
[20:07:38]   Epoch   1 | Train 0.0280 | Val 0.0275 | R2_roi -0.087 | R2_graph -0.257 | LR 3.0e-03
[20:07:53]   Epoch  20 | Train 0.0117 | Val 0.0138 | R2_roi 0.457 | R2_graph 0.974 | LR 2.7e-03
[20:08:09]   Epoch  40 | Train 0.0096 | Val 0.0110 | R2_roi 0.568 | R2_graph 0.971 | LR 2.0e-03
[20:08:26]   Epoch  60 | Train 0.0091 | Val 0.0101 | R2_roi 0.600 | R2_grap

# 8 — HC Data Loading and Split

In [8]:
# =============================================================================
# S8 --- HC DATA + SPLIT + HOLDOUT
# =============================================================================

log.info('=' * 70)
log.info('STAGE 2: HC DATA LOADING')
log.info('=' * 70)

hc_graphs, hc_file_info = load_hc_graphs(
    roi_meta, network_assignment, N_NETWORKS, sc_matrix, include_mvar=True)
hc_train_graphs, hc_test_graphs = split_hc_by_subject(hc_graphs, hc_file_info)
hc_train_graphs, hc_holdout_graphs = train_test_split(
    hc_train_graphs, test_size=HC_HOLDOUT_FRAC, random_state=SEED)
log.info('HC split: %d train, %d test (within-subject), %d holdout (unseen subjects)',
         len(hc_train_graphs), len(hc_test_graphs), len(hc_holdout_graphs))

[20:09:00] ======================================================================
[20:09:00] STAGE 2: HC DATA LOADING
[20:09:00] ======================================================================
[20:09:00] HC MVAR loaded: 295 matrices
[20:09:02] HC UKF from ch5 key: hc_all
[20:09:02] HC CSVs: 295
[20:09:02] HC files parsed: 295 (subjects: 30)
[20:09:02] HC ROI columns matched: 216 / 216
[20:09:14]   Built 50 / 295 HC graphs
[20:09:26]   Built 100 / 295 HC graphs
[20:09:38]   Built 150 / 295 HC graphs
[20:09:49]   Built 200 / 295 HC graphs
[20:10:00]   Built 250 / 295 HC graphs
[20:10:10] HC graphs: 295 / 295
[20:10:10] HC split: train=235 (24 subj), test=60 (6 subj)
[20:10:10] HC split: 199 train, 60 test (within-subject), 36 holdout (unseen subjects)


# 9 — MDD Empirical Graphs + Derived Feature Augmentation

All graphs receive 4 additional connectivity-derived features as reconstruction
targets: PLV node strength, MVAR in-strength, MVAR out-strength, within-network
PLV. These shape z during training but are excluded from the anomaly score.

In [9]:
# =============================================================================
# S9 --- MDD GRAPHS + AUGMENTATION
# =============================================================================

log.info('=' * 70)
log.info('MDD EMPIRICAL GRAPH ASSEMBLY')
log.info('=' * 70)

empirical_graphs, subjects_list, groups_map = build_empirical_graphs(
    ukf_df, roi_meta, plv_all, mvar_all, sc_matrix)

for key in list(EXCLUDED_SESSIONS):
    if key in empirical_graphs:
        del empirical_graphs[key]
        log.info('EXCLUDED: %s %s (known corrupted signal)', key[0], key[1])
log.info('MDD graphs after exclusion: %d', len(empirical_graphs))


# ── Derived feature augmentation ──
def augment_graph(graph, net_assign, n_rois=216):
    """Add connectivity-derived features as reconstruction targets."""
    physics = graph.x[:, :3]
    n = graph.x.size(0)

    def _ea(g, rel, ne):
        ea = getattr(g, f'edge_attr_{rel}', None)
        if ea is None: return torch.ones(ne)
        ea = ea.float().view(-1)
        return ea if ea.shape[0] == ne else torch.ones(ne)

    s_plv = torch.zeros(n)
    if hasattr(graph, 'edge_index_plv') and graph.edge_index_plv.numel() > 0:
        ei = graph.edge_index_plv
        s_plv.scatter_add_(0, ei[0], _ea(graph, 'plv', ei.size(1)))
        s_plv /= max(n, 1)

    s_in, s_out = torch.zeros(n), torch.zeros(n)
    if hasattr(graph, 'edge_index_mvar') and graph.edge_index_mvar.numel() > 0:
        ei = graph.edge_index_mvar
        ea = _ea(graph, 'mvar', ei.size(1)).abs()
        s_in.scatter_add_(0, ei[1], ea); s_out.scatter_add_(0, ei[0], ea)
        s_in /= max(n, 1); s_out /= max(n, 1)

    plv_w = torch.zeros(n)
    if hasattr(graph, 'edge_index_plv') and graph.edge_index_plv.numel() > 0 and net_assign is not None:
        ei = graph.edge_index_plv
        ea = _ea(graph, 'plv', ei.size(1))
        na = net_assign[:n]
        same = (na[ei[0]] == na[ei[1]])
        ws, wc = torch.zeros(n), torch.zeros(n)
        ws.scatter_add_(0, ei[0][same], ea[same])
        wc.scatter_add_(0, ei[0][same], torch.ones(same.sum()))
        plv_w = ws / wc.clamp(min=1)

    graph.recon_target = torch.cat([
        physics, s_plv.unsqueeze(1), s_in.unsqueeze(1),
        s_out.unsqueeze(1), plv_w.unsqueeze(1),
    ], dim=1)
    return graph


for g in hc_train_graphs + hc_test_graphs + hc_holdout_graphs:
    augment_graph(g, network_assignment)
for g in empirical_graphs.values():
    augment_graph(g, network_assignment)
log.info('All graphs augmented with 7-feature reconstruction targets')

[20:10:10] ======================================================================
[20:10:10] MDD EMPIRICAL GRAPH ASSEMBLY
[20:10:10] ======================================================================
[20:10:11] MDD graphs: 38 (subjects: 19, groups: {'active': 11, 'sham': 8})
[20:10:11] EXCLUDED: LS_E4209 rest2 (known corrupted signal)
[20:10:11] MDD graphs after exclusion: 37
[20:10:11] All graphs augmented with 7-feature reconstruction targets


# 10 — HopfGAE v6: Coupled Edge-Node Bottleneck

The single architectural difference from v5: edge decoders receive **z** (trainable,
6-dim) via concatenation [z_i ∥ z_j], not h (frozen, 32-dim) via |h_i − h_j|. This
creates a gradient path from the edge loss through z into fc_z, forcing the
bottleneck to encode connectivity structure alongside dynamics.

In [10]:
# =============================================================================
# S10 --- HopfGAE v6 MODEL DEFINITION
# =============================================================================


class HopfGAEv6(nn.Module):
    """
    Denoising Graph Autoencoder with coupled edge-node bottleneck.

    Encoder (frozen): GAT → h (32-dim)
    Bottleneck:       h → Linear → z (6-dim, trainable)
    Node decoder:     Dropout(z) → Linear → 7 features
    Edge decoders:    MLP([z_i ∥ z_j]) → P(edge) per relation  ← coupled through z
    Denoising:        x_noisy = x + σ·N(0,I) during training
    """

    def __init__(self, encoder_model, latent_dim=6, n_features_out=7,
                 noise_sigma=0.1, z_dropout=0.3):
        super().__init__()
        self.encoder = encoder_model
        for p in self.encoder.parameters():
            p.requires_grad = False

        hd = encoder_model.hidden_dim
        self.fc_z = nn.Linear(hd, latent_dim)
        self.z_dropout = nn.Dropout(p=z_dropout)
        self.node_decoder = nn.Linear(latent_dim, n_features_out)

        # Edge decoders on z (coupled): [z_i || z_j] preserves directed asymmetry
        eh = max(latent_dim, 8)
        self.edge_decoders = nn.ModuleDict({
            rel: nn.Sequential(nn.Linear(2 * latent_dim, eh), nn.ELU(), nn.Linear(eh, 1))
            for rel in ["plv", "sc", "mvar"]
        })

        self.latent_dim = latent_dim
        self.hidden_dim = hd
        self.noise_sigma = noise_sigma
        self.n_features_out = n_features_out

    def encode(self, data):
        if not hasattr(data, 'batch') or data.batch is None:
            data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=data.x.device)
        with torch.no_grad():
            h = self.encoder(data)['node_embeddings']
        return h, self.fc_z(h), data.batch

    def decode_nodes(self, z):
        return self.node_decoder(self.z_dropout(z))

    def decode_edges(self, z, edge_index, relation):
        if relation not in self.edge_decoders:
            return None
        src, dst = edge_index
        return torch.sigmoid(self.edge_decoders[relation](
            torch.cat([z[src], z[dst]], dim=-1)).squeeze(-1))

    def forward(self, data):
        if self.training and self.noise_sigma > 0:
            data = data.clone()
            data.x = data.x + self.noise_sigma * torch.randn_like(data.x)
        h, z, batch = self.encode(data)
        return {'node_recon': self.decode_nodes(z), 'h': h, 'z': z, 'batch': batch}


class GAELossV6(nn.Module):
    """Loss with coupled edge reconstruction: edge decoders receive z (not h)."""

    def __init__(self, lambda_edge=0.5, lambda_graph=0.1, feature_weights=None):
        super().__init__()
        self.lambda_edge = lambda_edge
        self.lambda_graph = lambda_graph
        if feature_weights is not None:
            self.register_buffer('fw', feature_weights)
        else:
            self.register_buffer('fw', RECON_FEATURE_WEIGHTS)

    def forward(self, result, data, gae_model=None):
        ld = {}
        target, recon = data.recon_target, result['node_recon']
        w = self.fw.to(recon.device)
        if w.shape[0] < recon.shape[1]:
            w = torch.cat([w, torch.ones(recon.shape[1] - w.shape[0], device=w.device)])
        l_node = ((recon - target) ** 2 * w[:recon.shape[1]]).mean()
        ld['node_recon'] = l_node.item()

        batch = result['batch']
        ar, at = recon[:, 0], target[:, 0]
        mr = global_mean_pool(ar.unsqueeze(1), batch).squeeze(1)
        mt = global_mean_pool(at.unsqueeze(1), batch).squeeze(1)
        vr = torch.clamp(global_mean_pool((ar**2).unsqueeze(1), batch).squeeze(1) - mr**2, min=1e-8)
        vt = torch.clamp(global_mean_pool((at**2).unsqueeze(1), batch).squeeze(1) - mt**2, min=1e-8)
        l_graph = F.mse_loss(mr, mt) + F.mse_loss(vr.sqrt(), vt.sqrt())
        ld['graph_level'] = l_graph.item()

        z = result['z']  # coupled: edge decoders use z
        l_edge = torch.tensor(0.0, device=z.device)
        ne = 0
        if gae_model is not None:
            for rel in ['plv', 'sc', 'mvar']:
                ek = f'edge_index_{rel}'
                if hasattr(data, ek) and getattr(data, ek).numel() > 0:
                    ei = getattr(data, ek)
                    ep = gae_model.decode_edges(z, ei, rel)
                    if ep is not None:
                        l_edge = l_edge + F.binary_cross_entropy(ep, torch.ones_like(ep))
                        ne += 1
                        nn_ = z.size(0)
                        neg_ei = torch.stack([torch.randint(0, nn_, (min(ei.size(1), nn_*2),), device=z.device),
                                              torch.randint(0, nn_, (min(ei.size(1), nn_*2),), device=z.device)])
                        np_ = gae_model.decode_edges(z, neg_ei, rel)
                        if np_ is not None:
                            l_edge = l_edge + F.binary_cross_entropy(np_, torch.zeros_like(np_))
                            ne += 1
        if ne > 0:
            l_edge = l_edge / ne
        ld['edge_recon'] = l_edge.item()

        total = l_node + self.lambda_graph * l_graph + self.lambda_edge * l_edge
        ld['total'] = total.item()
        return total, ld


seed_everything(SEED)  # reset RNG for deterministic GAE weight initialization
gae = HopfGAEv6(encoder, latent_dim=LATENT_DIM, n_features_out=N_FEATURES_OUT,
                 noise_sigma=NOISE_SIGMA, z_dropout=Z_DROPOUT)

n_frozen = sum(p.numel() for p in gae.parameters() if not p.requires_grad)
n_trainable = sum(p.numel() for p in gae.parameters() if p.requires_grad)
log.info('HopfGAE v6 (coupled): frozen=%d, trainable=%d, d_z=%d',
         n_frozen, n_trainable, LATENT_DIM)
log.info('  Edge decoder input: [z_i || z_j] = %d dims (trainable, gradient-coupled)',
         2 * LATENT_DIM)

[20:10:11] HopfGAE v6 (coupled): frozen=5485, trainable=586, d_z=6
[20:10:11]   Edge decoder input: [z_i || z_j] = 12 dims (trainable, gradient-coupled)


# 11 — GAE Training

Denoising graph autoencoder trained on HC data only. Loss combines feature-weighted
node reconstruction (7 targets), graph-level mean/std of $a$, and per-relation
edge reconstruction through the coupled bottleneck ($\lambda_\text{edge} = 0.5$).

In [11]:
# =============================================================================
# S11 --- GAE TRAINING
# =============================================================================

log.info('=' * 70)
log.info('STAGE 2: GAE TRAINING (coupled edge-node bottleneck)')
log.info('=' * 70)

gae_criterion = GAELossV6(lambda_edge=LAMBDA_EDGE, lambda_graph=LAMBDA_GRAPH,
                           feature_weights=TRAIN_WEIGHTS)
trainable_params = [p for p in gae.parameters() if p.requires_grad]
gae_optimizer = torch.optim.AdamW(trainable_params, lr=GAE_LR, weight_decay=GAE_WD)
gae_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(gae_optimizer, T_max=GAE_EPOCHS)

gae_train_loader = PyGDataLoader(hc_train_graphs, batch_size=GAE_BS, shuffle=True)
gae_test_loader = PyGDataLoader(hc_test_graphs, batch_size=max(1, len(hc_test_graphs)))
gae_history = {'epoch': [], 'train_loss': [], 'test_loss': [],
               'node_recon': [], 'edge_recon': []}

for epoch in range(1, GAE_EPOCHS + 1):
    gae.train()
    el, en, ee, nb = 0., 0., 0., 0
    for bd in gae_train_loader:
        gae_optimizer.zero_grad()
        res = gae(bd)
        loss, ld = gae_criterion(res, bd, gae_model=gae)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        gae_optimizer.step()
        el += ld['total']; en += ld['node_recon']; ee += ld['edge_recon']; nb += 1
    gae_scheduler.step()

    gae.eval()
    with torch.no_grad():
        tl = sum(gae_criterion(gae(tb), tb, gae_model=gae)[0].item() for tb in gae_test_loader)

    gae_history['epoch'].append(epoch)
    gae_history['train_loss'].append(el / max(nb, 1))
    gae_history['test_loss'].append(tl)
    gae_history['node_recon'].append(en / max(nb, 1))
    gae_history['edge_recon'].append(ee / max(nb, 1))

    if epoch % 20 == 0 or epoch == 1:
        log.info('  Epoch %3d | Train %.5f | Test %.5f | Node %.5f | Edge %.5f',
                 epoch, el / max(nb, 1), tl, en / max(nb, 1), ee / max(nb, 1))

log.info('GAE training complete.')

torch.save(gae.state_dict(), RESULTS_V6 / 'hopf_gae_v6.pt')
pd.DataFrame(gae_history).to_csv(RESULTS_V6 / 'gae_train_history.csv', index=False)

[20:10:11] ======================================================================
[20:10:11] STAGE 2: GAE TRAINING (coupled edge-node bottleneck)
[20:10:11] ======================================================================
[20:10:13]   Epoch   1 | Train 0.61449 | Test 0.52771 | Node 0.24520 | Edge 0.69472
[20:10:47]   Epoch  20 | Train 0.36306 | Test 0.35323 | Node 0.01903 | Edge 0.68755
[20:11:23]   Epoch  40 | Train 0.35447 | Test 0.35028 | Node 0.01253 | Edge 0.68375
[20:12:00]   Epoch  60 | Train 0.35189 | Test 0.34973 | Node 0.01020 | Edge 0.68327
[20:12:36]   Epoch  80 | Train 0.35060 | Test 0.34948 | Node 0.00921 | Edge 0.68261
[20:13:12]   Epoch 100 | Train 0.34443 | Test 0.34285 | Node 0.00936 | Edge 0.66998
[20:13:49]   Epoch 120 | Train 0.33934 | Test 0.33807 | Node 0.00880 | Edge 0.66086
[20:14:25]   Epoch 140 | Train 0.33778 | Test 0.33729 | Node 0.00859 | Edge 0.65816
[20:15:02]   Epoch 160 | Train 0.33761 | Test 0.33628 | Node 0.00846 | Edge 0.65807
[20:15:38]   Epo

# Stage 3 — Anomaly Detection

### Physics-Only Scoring

The anomaly score uses reconstruction error on the first 3 features only
($a_j$, $\omega_j$, $\chi^2_j$) with weights [2, 1, 1]. The 4 connectivity-derived
features shaped $\mathbf{z}$ during training but are excluded from the score.
This eliminates the dilution problem (7-feature d=3.64 vs physics-only d=3.02)
while retaining the connectivity-informed latent representation.

In [17]:
# =============================================================================
# S12 --- ANOMALY SCORING (physics-only, label-free)
# =============================================================================

log.info('=' * 70)
log.info('STAGE 3: ANOMALY SCORING')
log.info('=' * 70)


def score_physics(model, graph, weights=SCORE_WEIGHTS):
    """Per-ROI reconstruction error on physics features only."""
    g_c = prepare_graph_for_batching(graph) if hasattr(graph, 'subject') else graph
    model.eval()
    with torch.no_grad():
        res = model(g_c)
    nf = len(weights)
    target = g_c.recon_target[:, :nf].numpy()
    recon = res['node_recon'][:, :nf].numpy()
    return np.average(np.clip(target - recon, -1e3, 1e3) ** 2, axis=1, weights=weights.numpy())


def score_batch(model, graphs, label=''):
    """Score a batch, return per-graph means and per-ROI arrays."""
    scores, rois = [], []
    for g in graphs:
        err = score_physics(model, g)
        scores.append(err.mean())
        rois.append(err)
    if label:
        log.info('  %s: %.6f +/- %.6f (n=%d)', label, np.mean(scores), np.std(scores), len(scores))
    return np.array(scores), rois


# Score everything
hc_tr_scores, hc_tr_roi = score_batch(gae, hc_train_graphs, 'HC train')
hc_te_scores, hc_te_roi = score_batch(gae, hc_test_graphs, 'HC test')

# HC norms
hc_all_scores = np.concatenate([hc_tr_scores, hc_te_scores])
hc_mu, hc_sd = hc_all_scores.mean(), max(hc_all_scores.std(), 1e-12)
log.info('  HC norms: mu=%.6f sd=%.6f', hc_mu, hc_sd)

# MDD scoring
mdd_scores, mdd_roi_errors = {}, {}
for key, g in empirical_graphs.items():
    err = score_physics(gae, g)
    mdd_scores[key] = err.mean()
    mdd_roi_errors[key] = err

mdd_r1 = np.array([mdd_scores[(s, 'rest1')] for s in subjects_list
                    if (s, 'rest1') in mdd_scores])
log.info('  MDD rest1: %.6f +/- %.6f (n=%d)', mdd_r1.mean(), mdd_r1.std(), len(mdd_r1))

# Z-score
hc_te_z = (hc_te_scores - hc_mu) / hc_sd
mdd_r1_z = (mdd_r1 - hc_mu) / hc_sd
log.info('  HC test z: %.4f +/- %.4f', hc_te_z.mean(), hc_te_z.std())
log.info('  MDD rest1 z: %.4f +/- %.4f', mdd_r1_z.mean(), mdd_r1_z.std())

# Outlier detection
outlier_threshold = hc_mu + (OUTLIER_N_SD + 1) * hc_sd
outlier_subjects = set()
for subj in subjects_list:
    for sess in ['rest1', 'rest2']:
        key = (subj, sess)
        if key in mdd_scores and mdd_scores[key] > outlier_threshold:
            log.info('  OUTLIER: %s %s score=%.4f', subj, sess, mdd_scores[key])
            outlier_subjects.add(subj)
analysis_subjects = [s for s in subjects_list if s not in outlier_subjects]
log.info('Outlier threshold (HC_mean + %d×SD): %.6f', OUTLIER_N_SD, outlier_threshold)
log.info('Outliers: %d / %d', len(outlier_subjects), len(subjects_list))
log.info('Analysis subjects: %d', len(analysis_subjects))

# HC holdout false positive rate
hc_ho_scores, _ = score_batch(gae, hc_holdout_graphs, 'HC holdout')
n_fp = (hc_ho_scores > outlier_threshold).sum()
log.info('HC holdout false positive rate: %d/%d (%.1f%%)',
         n_fp, len(hc_ho_scores), 100 * n_fp / len(hc_ho_scores))

# Export scores
export_rows = []
for i, s in enumerate(hc_te_z):
    export_rows.append({'cohort': 'HC_test', 'group': 'HC', 'session': 'avg', 'score': float(s)})
for i, subj in enumerate(subjects_list):
    if i < len(mdd_r1_z):
        export_rows.append({'cohort': 'MDD', 'group': groups_map.get(subj, ''),
                            'session': 'rest1', 'subject': subj, 'score': float(mdd_r1_z[i])})
pd.DataFrame(export_rows).to_csv(RESULTS_V6 / 'anomaly_scores_v6.csv', index=False)

[20:19:25] ======================================================================
[20:19:25] STAGE 3: ANOMALY SCORING
[20:19:25] ======================================================================
[20:19:26]   HC train: 0.012030 +/- 0.003875 (n=199)
[20:19:27]   HC test: 0.012100 +/- 0.002682 (n=60)
[20:19:27]   HC norms: mu=0.012046 sd=0.003634
[20:19:27]   MDD rest1: 0.022818 +/- 0.004238 (n=19)
[20:19:27]   HC test z: 0.0148 +/- 0.7379
[20:19:27]   MDD rest1 z: 2.9642 +/- 1.1662
[20:19:27]   OUTLIER: SA_E4051 rest1 score=0.0350
[20:19:27] Outlier threshold (HC_mean + 5×SD): 0.033850
[20:19:27] Outliers: 1 / 19
[20:19:27] Analysis subjects: 18
[20:19:27]   HC holdout: 0.012843 +/- 0.004523 (n=36)
[20:19:27] HC holdout false positive rate: 0/36 (0.0%)


# 13 — Statistical Analysis

In [18]:
# =============================================================================
# S13 --- STATISTICAL ANALYSIS
# =============================================================================

log.info('=' * 70)
log.info('STATISTICAL ANALYSIS')
log.info('=' * 70)


def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / max(np.sqrt((np.var(a) + np.var(b)) / 2), 1e-12)


# ── 1. HC vs MDD Separation ──
log.info('')
log.info('=== 1. HC vs MDD ANOMALY SEPARATION ===')
d_sep = cohens_d(mdd_r1_z, hc_te_z)
t_val, p_welch = sp_stats.ttest_ind(mdd_r1_z, hc_te_z, equal_var=False)
u_stat, p_mwu = sp_stats.mannwhitneyu(mdd_r1_z, hc_te_z, alternative='greater')
log.info('  HC test:  %.4f +/- %.4f (n=%d)', hc_te_z.mean(), hc_te_z.std(), len(hc_te_z))
log.info('  MDD rest1: %.4f +/- %.4f (n=%d)', mdd_r1_z.mean(), mdd_r1_z.std(), len(mdd_r1_z))
log.info('  Cohen d: %+.3f', d_sep)
log.info('  Welch t=%.3f, p=%.4e', t_val, p_welch)
log.info('  Mann-Whitney U=%.0f, p=%.4e', u_stat, p_mwu)

# Permutation null
rng = np.random.RandomState(SEED)
combined = np.concatenate([hc_te_z, mdd_r1_z])
n_hc = len(hc_te_z)
null_dist = np.zeros(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    idx = rng.permutation(len(combined))
    null_dist[i] = cohens_d(combined[idx[n_hc:]], combined[idx[:n_hc]])
perm_p = (np.abs(null_dist) >= np.abs(d_sep)).mean()
log.info('  Permutation null (n=%d): p=%.6f', N_PERMUTATIONS, perm_p)

# Holdout vs MDD
hc_ho_z = (hc_ho_scores - hc_mu) / hc_sd
d_ho = cohens_d(mdd_r1_z, hc_ho_z)
t_ho, p_ho = sp_stats.ttest_ind(mdd_r1_z, hc_ho_z, equal_var=False)
log.info('  HC holdout vs MDD: d=%+.3f, p=%.4e', d_ho, p_ho)

# ── 2. Overfitting check ──
log.info('')
log.info('=== 2. HC TRAIN vs TEST (overfitting check) ===')
hc_tr_z = (hc_tr_scores - hc_mu) / hc_sd
_, p_overfit = sp_stats.ttest_ind(hc_tr_z, hc_te_z, equal_var=False)
log.info('  Train: %.4f +/- %.4f | Test: %.4f +/- %.4f | p=%.4f',
         hc_tr_z.mean(), hc_tr_z.std(), hc_te_z.mean(), hc_te_z.std(), p_overfit)

# ── 3. Multi-scale intervention ──
log.info('')
n_act = sum(1 for s in analysis_subjects if groups_map.get(s, '').lower() in ('active', 'experimental'))
n_sha = len(analysis_subjects) - n_act
log.info('=== 3. MULTI-SCALE INTERVENTION (n=%d: %d active, %d sham) ===',
         len(analysis_subjects), n_act, n_sha)

scales = {'whole_brain': np.ones(len(roi_meta), dtype=bool),
          'circuit': circuit_mask, 'limbic': limbic_mask, 'subcortical': subcort_mask}
intervention_results = {}

for sn, mask in scales.items():
    act_d, sha_d = [], []
    for subj in analysis_subjects:
        r1k, r2k = (subj, 'rest1'), (subj, 'rest2')
        if r1k not in mdd_roi_errors or r2k not in mdd_roi_errors:
            continue
        delta = mdd_roi_errors[r2k][mask].mean() - mdd_roi_errors[r1k][mask].mean()
        grp = groups_map.get(subj, '').lower()
        (act_d if grp in ('active', 'experimental') else sha_d).append(delta)

    act_d, sha_d = np.array(act_d), np.array(sha_d)
    if len(act_d) < 2 or len(sha_d) < 2:
        continue

    d_int = cohens_d(act_d, sha_d)
    t_int, p_int = sp_stats.ttest_ind(act_d, sha_d, equal_var=False)

    comb = np.concatenate([act_d, sha_d]); na = len(act_d)
    rng_p = np.random.RandomState(SEED)
    perm_d = np.zeros(N_PERMS)
    for pi in range(N_PERMS):
        sh = rng_p.permutation(comb)
        perm_d[pi] = cohens_d(sh[:na], sh[na:])
    p_perm = (np.abs(perm_d) >= np.abs(d_int)).mean()

    # Bootstrap CI
    rng_b = np.random.RandomState(SEED + 1)
    boot_d = np.zeros(N_BOOT)
    for bi in range(N_BOOT):
        boot_d[bi] = cohens_d(act_d[rng_b.randint(0, len(act_d), len(act_d))],
                              sha_d[rng_b.randint(0, len(sha_d), len(sha_d))])
    ci = np.percentile(boot_d, [2.5, 97.5])

    dir_act = 'TOWARD' if act_d.mean() < 0 else 'AWAY'
    dir_sha = 'TOWARD' if sha_d.mean() < 0 else 'AWAY'

    intervention_results[sn] = {'d': d_int, 'p': p_int, 'perm_p': p_perm,
                                'ci_lo': ci[0], 'ci_hi': ci[1]}
    log.info('  %-15s (%3d ROIs): d=%+.3f [%.2f, %.2f]  p=%.4f  perm_p=%.4f',
             sn, mask.sum(), d_int, ci[0], ci[1], p_int, p_perm)
    log.info('    Active: %+.6f (%s HC) | Sham: %+.6f (%s HC)',
             act_d.mean(), dir_act, sha_d.mean(), dir_sha)

# FDR correction
if intervention_results:
    raw_ps = [intervention_results[s]['perm_p'] for s in intervention_results]
    _, fdr_ps, _, _ = multipletests(raw_ps, method='fdr_bh')
    log.info('')
    log.info('  FDR-corrected permutation p-values:')
    for (sn, _), fp in zip(intervention_results.items(), fdr_ps):
        log.info('    %-15s raw=%.4f  FDR=%.4f %s', sn, intervention_results[sn]['perm_p'],
                 fp, '*' if fp < 0.05 else '')

# ── 4. Spatial heterogeneity ──
log.info('')
log.info('=== 4. SPATIAL HETEROGENEITY (delta SD of raw a) ===')
het_act, het_sha = [], []
het_act_c, het_sha_c = [], []
for subj in subjects_list:
    r1k, r2k = (subj, 'rest1'), (subj, 'rest2')
    if r1k not in empirical_graphs or r2k not in empirical_graphs:
        continue
    a1 = empirical_graphs[r1k].x[:, 0].numpy()
    a2 = empirical_graphs[r2k].x[:, 0].numpy()
    ds = np.std(a2) - np.std(a1)
    dsc = np.std(a2[circuit_mask]) - np.std(a1[circuit_mask])
    grp = groups_map.get(subj, '').lower()
    if grp in ('active', 'experimental'):
        het_act.append(ds); het_act_c.append(dsc)
    else:
        het_sha.append(ds); het_sha_c.append(dsc)

if len(het_act) >= 2 and len(het_sha) >= 2:
    for label, a, s in [('Whole-brain', het_act, het_sha), ('Circuit', het_act_c, het_sha_c)]:
        a, s = np.array(a), np.array(s)
        d_h = cohens_d(a, s)
        t_h, p_h = sp_stats.ttest_ind(a, s, equal_var=False)
        log.info('  [%s] d=%+.3f  t=%.3f  p=%.4f  (UKF target: +1.006)', label, d_h, t_h, p_h)

# ── 5. Circuit-specific anomaly map ──
log.info('')
log.info('=== 5. CIRCUIT-SPECIFIC ANOMALY MAP ===')
mdd_r1_roi_list = [mdd_roi_errors[(s, 'rest1')] for s in subjects_list
                   if (s, 'rest1') in mdd_roi_errors]
mean_roi_err = np.mean(mdd_r1_roi_list, axis=0) if mdd_r1_roi_list else None

if mean_roi_err is not None:
    top_idx = np.argsort(mean_roi_err)[::-1]
    log.info('  Top 15 anomalous ROIs:')
    for rank, idx in enumerate(top_idx[:15]):
        log.info('    %-50s %s %12.6f %s',
                 roi_meta.iloc[idx]['roi_name'], roi_meta.iloc[idx]['network'],
                 mean_roi_err[idx], '[CIRCUIT]' if circuit_mask[idx] else '')

    log.info('')
    log.info('  Network-level anomaly:')
    for net in sorted(roi_meta['network'].unique()):
        m = (roi_meta['network'] == net).values
        log.info('    %-25s %.6f', net, mean_roi_err[m].mean())

    log.info('')
    log.info('  Circuit enrichment (hypergeometric):')
    from scipy.stats import hypergeom
    for k in [10, 15, 20, 30]:
        top_k = top_idx[:k]
        K = circuit_mask.sum()
        n_circ = circuit_mask[top_k].sum()
        p_hyp = hypergeom.sf(n_circ - 1, len(roi_meta), K, k)
        enr = (n_circ / k) / (K / len(roi_meta))
        log.info('    Top %d: %d/%d (%.0f%%), enrichment = %.2fx, hypergeom p=%.4f %s',
                 k, n_circ, k, 100 * n_circ / k, enr, p_hyp, '*' if p_hyp < 0.05 else '')

    # Save ROI map
    roi_map = roi_meta[['roi_name', 'network', 'is_depression_circuit']].copy()
    roi_map['mean_anomaly'] = mean_roi_err
    roi_map.sort_values('mean_anomaly', ascending=False).to_csv(
        RESULTS_V6 / 'roi_anomaly_map_v6.csv', index=False)

# ── 6. Per-feature decomposition ──
log.info('')
log.info('=== 6. PER-FEATURE DECOMPOSITION ===')
feature_names = ['a', 'omega', 'chisq', 's_plv', 's_mvar_in', 's_mvar_out', 'plv_within']
mdd_gs = [empirical_graphs[(s, 'rest1')] for s in subjects_list if (s, 'rest1') in empirical_graphs]
gae.eval()
with torch.no_grad():
    for fi, fn in enumerate(feature_names):
        hc_f = [((g.recon_target[:, fi].numpy() -
                  gae(prepare_graph_for_batching(g))['node_recon'][:, fi].cpu().numpy()) ** 2).mean()
                for g in hc_test_graphs]
        mdd_f = [((g.recon_target[:, fi].numpy() -
                   gae(prepare_graph_for_batching(g))['node_recon'][:, fi].cpu().numpy()) ** 2).mean()
                 for g in mdd_gs]
        d_f = cohens_d(np.array(mdd_f), np.array(hc_f))
        log.info('  %-15s d=%+.3f', fn, d_f)

# ── Final summary ──
log.info('')
log.info('=' * 70)
log.info('ANALYSIS COMPLETE')
log.info('=' * 70)
log.info('Excluded: %d subjects by outlier detection', len(outlier_subjects))
log.info('Analysis: %d subjects (%d active, %d sham)', len(analysis_subjects), n_act, n_sha)

[20:19:27] ======================================================================
[20:19:27] STATISTICAL ANALYSIS
[20:19:27] ======================================================================
[20:19:27] 
[20:19:27] === 1. HC vs MDD ANOMALY SEPARATION ===
[20:19:27]   HC test:  0.0148 +/- 0.7379 (n=60)
[20:19:27]   MDD rest1: 2.9642 +/- 1.1662 (n=19)
[20:19:27]   Cohen d: +3.022
[20:19:27]   Welch t=10.129, p=7.3404e-10
[20:19:27]   Mann-Whitney U=1126, p=9.3276e-11
[20:19:27]   Permutation null (n=10000): p=0.000000
[20:19:27]   HC holdout vs MDD: d=+2.276, p=1.2902e-09
[20:19:27] 
[20:19:27] === 2. HC TRAIN vs TEST (overfitting check) ===
[20:19:27]   Train: -0.0045 +/- 1.0664 | Test: 0.0148 +/- 0.7379 | p=0.8750
[20:19:27] 
[20:19:27] === 3. MULTI-SCALE INTERVENTION (n=18: 11 active, 7 sham) ===
[20:19:28]   whole_brain     (216 ROIs): d=+1.305 [0.38, 2.89]  p=0.0167  perm_p=0.0325
[20:19:28]     Active: +0.001982 (AWAY HC) | Sham: -0.004325 (TOWARD HC)
[20:19:28]   circuit      

# Report Figures

# F0 — Figure Configuration

In [19]:
# =============================================================================
# F0 --- FIGURE SETUP
# =============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

IMG = RESULTS_V6 / 'img'
IMG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'font.size': 10,
                     'axes.titlesize': 11, 'figure.facecolor': 'white'})

# F1–F15 — All Report Figures

In [20]:
# =============================================================================
# F1 --- PRE-TRAINING CURVES
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(pretrain_history['epoch'], pretrain_history['train_loss'], label='Train')
ax1.plot(pretrain_history['epoch'], pretrain_history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Pre-Training Loss'); ax1.legend()
ax2.plot(pretrain_history['epoch'], pretrain_history['r2_graph'], color='#55A868', label='R² (graph-mean)')
ax2.plot(pretrain_history['epoch'], pretrain_history['r2'], color='#4C72B0', label='R² (per-ROI)', alpha=0.7)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('R²'); ax2.set_title('Bifurcation Parameter Recovery'); ax2.legend()
fig.tight_layout(); fig.savefig(IMG / 'fig_pretrain.png'); fig.savefig(IMG / 'fig_pretrain.pdf')
plt.close(fig); log.info('-> fig_pretrain')

# =============================================================================
# F2 --- GAE TRAINING CURVES
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gae_history['epoch'], gae_history['train_loss'], label='Train')
ax1.plot(gae_history['epoch'], gae_history['test_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('GAE Total Loss'); ax1.legend()
ax2.plot(gae_history['epoch'], gae_history['node_recon'], label='Node', color='#4C72B0')
ax2.plot(gae_history['epoch'], gae_history['edge_recon'], label='Edge', color='#DD8452')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Node vs Edge Reconstruction'); ax2.legend()
fig.tight_layout(); fig.savefig(IMG / 'fig_gae_training.png'); fig.savefig(IMG / 'fig_gae_training.pdf')
plt.close(fig); log.info('-> fig_gae_training')

# =============================================================================
# F3 --- HC vs MDD ANOMALY DISTRIBUTIONS
# =============================================================================

fig, ax = plt.subplots(figsize=(7, 4.5))
for data, pos, color in [(hc_te_z, 0, '#4C72B0'), (mdd_r1_z, 1, '#DD8452')]:
    parts = ax.violinplot([data], positions=[pos], showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color); pc.set_alpha(0.7)
ax.set_xticks([0, 1])
ax.set_xticklabels([f'HC (n={len(hc_te_z)})', f'MDD (n={len(mdd_r1_z)})'])
ax.set_ylabel('Anomaly Score (z-scored)')
ax.set_title(f'HC vs MDD Anomaly Score Distribution\n'
             f'Cohen\'s d = {d_sep:+.3f}, p = {p_welch:.2e} (Welch), perm p = {perm_p:.4f}')
fig.tight_layout(); fig.savefig(IMG / 'fig_hc_mdd.png'); fig.savefig(IMG / 'fig_hc_mdd.pdf')
plt.close(fig); log.info('-> fig_hc_mdd')

# =============================================================================
# F4 --- PERMUTATION NULL DISTRIBUTION
# =============================================================================

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_dist, bins=80, density=True, color='#999', alpha=0.7, label='Null')
ax.axvline(d_sep, color='#DD8452', lw=2, ls='--', label=f'Observed d={d_sep:+.3f}')
ax.axvline(-d_sep, color='#DD8452', lw=2, ls='--', alpha=0.4)
ax.set_xlabel("Cohen's d"); ax.set_ylabel('Density')
ax.set_title(f'Permutation Null (n={N_PERMUTATIONS}), p={perm_p:.6f}'); ax.legend()
fig.tight_layout(); fig.savefig(IMG / 'fig_perm_null.png'); fig.savefig(IMG / 'fig_perm_null.pdf')
plt.close(fig); log.info('-> fig_perm_null')

# =============================================================================
# F5 --- MULTI-SCALE INTERVENTION EFFECT
# =============================================================================

if intervention_results:
    scale_names = list(intervention_results.keys())
    ds = [intervention_results[s]['d'] for s in scale_names]
    ps = [intervention_results[s]['perm_p'] for s in scale_names]
    cis = [(intervention_results[s]['ci_lo'], intervention_results[s]['ci_hi']) for s in scale_names]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    bars = ax.bar(range(len(scale_names)), ds, color=colors[:len(scale_names)],
                  yerr=[[d - ci[0] for d, ci in zip(ds, cis)],
                        [ci[1] - d for d, ci in zip(ds, cis)]],
                  capsize=4, error_kw={'lw': 1.5})
    ax.set_xticks(range(len(scale_names))); ax.set_xticklabels(scale_names, rotation=15)
    ax.set_ylabel("Cohen's d (active − sham)")
    ax.set_title('Multi-Scale Intervention Effect (Δanomaly, physics-only scoring)')
    for i, (d, p) in enumerate(zip(ds, ps)):
        ax.text(i, d + 0.15, f'd={d:+.2f}\np={p:.3f}{"*" if p < 0.05 else ""}',
                ha='center', va='bottom', fontsize=8)
    fig.tight_layout(); fig.savefig(IMG / 'fig_intervention.png'); fig.savefig(IMG / 'fig_intervention.pdf')
    plt.close(fig); log.info('-> fig_intervention')

# =============================================================================
# F6 --- CIRCUIT ENRICHMENT (top 30 ROIs)
# =============================================================================

if mean_roi_err is not None:
    ti = np.argsort(mean_roi_err)[::-1][:30]
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ['#C44E52' if circuit_mask[i] else '#999' for i in ti]
    names = [roi_meta.iloc[i]['roi_name'][:40] for i in ti]
    ax.barh(range(len(ti)), [mean_roi_err[i] for i in ti], color=colors)
    ax.set_yticks(range(len(ti))); ax.set_yticklabels(names, fontsize=7); ax.invert_yaxis()
    ax.set_xlabel('Mean Anomaly Score (physics-only)')
    ax.set_title('Top 30 Anomalous ROIs (red = depression circuit)')
    fig.tight_layout(); fig.savefig(IMG / 'fig_enrichment_bar.png'); fig.savefig(IMG / 'fig_enrichment_bar.pdf')
    plt.close(fig); log.info('-> fig_enrichment_bar')

# =============================================================================
# F7 --- ENRICHMENT CURVE
# =============================================================================

if mean_roi_err is not None:
    from scipy.stats import hypergeom
    enr_data = []
    top_idx = np.argsort(mean_roi_err)[::-1]
    K = circuit_mask.sum()
    for k in range(5, 101, 5):
        nc = circuit_mask[top_idx[:k]].sum()
        enr_data.append({'k': k, 'enrichment': (nc / k) / (K / len(roi_meta)),
                         'p': hypergeom.sf(nc - 1, len(roi_meta), K, k)})
    enr_df = pd.DataFrame(enr_data)
    enr_df.to_csv(RESULTS_V6 / 'enrichment_curve.csv', index=False)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
    a1.plot(enr_df['k'], enr_df['enrichment'], 'o-', color='#C44E52')
    a1.axhline(1, color='gray', ls='--', alpha=0.5); a1.set_xlabel('Top-k'); a1.set_ylabel('Enrichment (×)')
    a1.set_title('Depression Circuit Enrichment')
    a2.semilogy(enr_df['k'], enr_df['p'], 'o-', color='#4C72B0')
    a2.axhline(0.05, color='gray', ls='--', alpha=0.5, label='p=0.05')
    a2.set_xlabel('Top-k'); a2.set_ylabel('Hypergeometric p'); a2.set_title('Enrichment Significance'); a2.legend()
    fig.tight_layout(); fig.savefig(IMG / 'fig_enrichment_curve.png'); fig.savefig(IMG / 'fig_enrichment_curve.pdf')
    plt.close(fig); log.info('-> fig_enrichment_curve')

# =============================================================================
# F8 --- NETWORK-LEVEL ANOMALY BREAKDOWN
# =============================================================================

if mean_roi_err is not None:
    net_means = [(net, mean_roi_err[(roi_meta['network'] == net).values].mean())
                 for net in sorted(roi_meta['network'].unique())]
    net_means.sort(key=lambda x: -x[1])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(net_means)), [x[1] for x in net_means],
           color=['#C44E52' if 'Limbic' in x[0] else '#4C72B0' for x in net_means])
    ax.set_xticks(range(len(net_means)))
    ax.set_xticklabels([x[0] for x in net_means], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Mean Node Anomaly'); ax.set_title('Network-Level Anomaly Breakdown')
    fig.tight_layout(); fig.savefig(IMG / 'fig_network_anomaly.png'); fig.savefig(IMG / 'fig_network_anomaly.pdf')
    plt.close(fig); log.info('-> fig_network_anomaly')

# =============================================================================
# F9 --- ROI HEATMAP SORTED BY NETWORK
# =============================================================================

if mean_roi_err is not None:
    sidx = roi_meta.sort_values(['network', 'roi_name']).index
    serr = mean_roi_err[sidx]
    scirc = circuit_mask[sidx]
    snet = roi_meta.iloc[sidx]['network'].values

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(serr)), serr,
           color=['#C44E52' if c else '#4C72B0' for c in scirc], width=1.0, edgecolor='none')
    prev = None
    for i, n in enumerate(snet):
        if n != prev:
            if prev is not None: ax.axvline(i - 0.5, color='k', lw=0.5, alpha=0.3)
            prev = n
    ax.set_xlabel('ROI (sorted by network)'); ax.set_ylabel('Mean Anomaly')
    ax.set_title('ROI-Level Anomaly (red = depression circuit)')
    fig.tight_layout(); fig.savefig(IMG / 'fig_roi_heatmap.png'); fig.savefig(IMG / 'fig_roi_heatmap.pdf')
    plt.close(fig); log.info('-> fig_roi_heatmap')

# =============================================================================
# F10 --- SPATIAL HETEROGENEITY
# =============================================================================

if len(het_act) >= 2:
    fig, ax = plt.subplots(figsize=(6, 4))
    ha, hs = np.array(het_act), np.array(het_sha)
    d_h = cohens_d(ha, hs); t_h, p_h = sp_stats.ttest_ind(ha, hs, equal_var=False)
    ax.bar([0, 1], [ha.mean(), hs.mean()], yerr=[ha.std(), hs.std()],
           color=['#DD8452', '#4C72B0'], capsize=5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Active', 'Sham'])
    ax.set_ylabel('ΔSD(a) rest2 − rest1')
    ax.set_title(f'Spatial Heterogeneity (raw a)\nd={d_h:+.3f}, p={p_h:.4f} (UKF target: +1.006)')
    fig.tight_layout(); fig.savefig(IMG / 'fig_heterogeneity.png'); fig.savefig(IMG / 'fig_heterogeneity.pdf')
    plt.close(fig); log.info('-> fig_heterogeneity')

# =============================================================================
# F11 --- CONVERGENT VALIDATION
# =============================================================================

fig, ax = plt.subplots(figsize=(8, 5))
methods = ['UKF (Ch4)', 'Perm Cluster (Ch5)', 'Hopf-GAE (v6)']
wb_d = [0.835, np.nan, abs(intervention_results.get('whole_brain', {}).get('d', np.nan))]
het_d_vals = [1.006, np.nan, abs(d_h) if 'd_h' in dir() else np.nan]
x = np.arange(len(methods)); w = 0.3
ax.bar(x - w/2, [abs(d) if not np.isnan(d) else 0 for d in wb_d], w,
       label='Intervention |d|', color='#DD8452')
ax.bar(x + w/2, [abs(d) if not np.isnan(d) else 0 for d in het_d_vals], w,
       label='Heterogeneity |d|', color='#55A868')
ax.set_xticks(x); ax.set_xticklabels(methods); ax.set_ylabel("|Cohen's d|")
ax.set_title('Convergent Validation'); ax.legend()
fig.tight_layout(); fig.savefig(IMG / 'fig_convergent.png'); fig.savefig(IMG / 'fig_convergent.pdf')
plt.close(fig); log.info('-> fig_convergent')

# =============================================================================
# F12 --- PARAMETER BUDGET
# =============================================================================

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie([n_frozen, n_trainable],
       labels=[f'Frozen encoder\n({n_frozen:,})', f'Trainable GAE\n({n_trainable:,})'],
       colors=['#4C72B0', '#DD8452'], autopct='%1.0f%%', startangle=90)
ax.set_title(f'Parameter Budget (Total: {n_frozen + n_trainable:,})')
fig.tight_layout(); fig.savefig(IMG / 'fig_params.png'); fig.savefig(IMG / 'fig_params.pdf')
plt.close(fig); log.info('-> fig_params')

# =============================================================================
# F13 --- UMAP LATENT SPACE
# =============================================================================

try:
    from umap import UMAP

    z_hc = np.array([gae(prepare_graph_for_batching(g))['z'].detach().numpy().mean(0)
                     for g in hc_test_graphs])
    mdd_gs = [empirical_graphs[(s, 'rest1')] for s in subjects_list if (s, 'rest1') in empirical_graphs]
    z_mdd = np.array([gae(prepare_graph_for_batching(g))['z'].detach().numpy().mean(0)
                      for g in mdd_gs])

    all_z = np.vstack([z_hc, z_mdd])
    labels = np.array(['HC'] * len(z_hc) + ['MDD'] * len(z_mdd))
    emb = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(all_z)

    umap_df = pd.DataFrame({'UMAP1': emb[:, 0], 'UMAP2': emb[:, 1], 'group': labels})
    umap_df.to_csv(RESULTS_V6 / 'umap_embeddings.csv', index=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    for grp, c, m in [('HC', '#4C72B0', 'o'), ('MDD', '#DD8452', 's')]:
        mask = umap_df['group'] == grp
        ax.scatter(umap_df.loc[mask, 'UMAP1'], umap_df.loc[mask, 'UMAP2'],
                   c=c, marker=m, s=40, alpha=0.7, label=grp, edgecolors='white', lw=0.3)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title('Latent Space (coupled v6)'); ax.legend()
    fig.tight_layout(); fig.savefig(IMG / 'fig_umap.png'); fig.savefig(IMG / 'fig_umap.pdf')
    plt.close(fig); log.info('-> fig_umap')
except ImportError:
    log.warning('UMAP not available.')

log.info('All figures saved to %s', IMG)

[20:19:34] -> fig_pretrain
[20:19:34] -> fig_gae_training
[20:19:34] -> fig_hc_mdd
[20:19:34] -> fig_perm_null
[20:19:34] -> fig_intervention
[20:19:34] -> fig_enrichment_bar
[20:19:35] -> fig_enrichment_curve
[20:19:35] -> fig_network_anomaly
[20:19:35] -> fig_roi_heatmap
[20:19:35] -> fig_heterogeneity
[20:19:35] -> fig_convergent
[20:19:35] -> fig_params
[20:19:36] -> fig_umap
[20:19:36] All figures saved to results/hopf_stgnn/v6/img


# 14 — Summary Statistics Export

In [21]:
# =============================================================================
# S14 --- SUMMARY EXPORT
# =============================================================================

log.info('=' * 70)
log.info('SUMMARY EXPORT')
log.info('=' * 70)

summary_rows = [
    {'test': 'HC vs MDD separation', 'scale': 'whole_brain',
     'd': d_sep, 'p': p_welch, 'method': 'Welch t (physics-only, coupled v6)', 'perm_p': perm_p},
]
for sn, iv in intervention_results.items():
    summary_rows.append({'test': f'Intervention ({sn})', 'scale': sn,
                         'd': iv['d'], 'p': iv['p'], 'method': 'Welch t + permutation',
                         'perm_p': iv['perm_p']})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_V6 / 'summary_statistics_v6.csv', index=False)
log.info('\n%s', summary_df.to_string(index=False))

log.info('')
log.info('=' * 70)
log.info('v6 COMPLETE')
log.info('=' * 70)
log.info('Artifacts: %s', RESULTS_V6)
log.info('  hopf_gae_v6.pt           -- GAE weights')
log.info('  gae_train_history.csv     -- Training curves')
log.info('  summary_statistics_v6.csv -- All effect sizes')
log.info('  anomaly_scores_v6.csv     -- Per-subject scores')
log.info('  roi_anomaly_map_v6.csv    -- Per-ROI ranking')
log.info('  enrichment_curve.csv      -- Continuous enrichment')
log.info('  umap_embeddings.csv       -- Latent embeddings')
log.info('  img/                      -- All figures (PNG + PDF)')

[20:19:36] ======================================================================
[20:19:36] SUMMARY EXPORT
[20:19:36] ======================================================================
[20:19:36] 
                      test       scale        d            p                             method  perm_p
      HC vs MDD separation whole_brain 3.022359 7.340397e-10 Welch t (physics-only, coupled v6)  0.0000
Intervention (whole_brain) whole_brain 1.304879 1.671539e-02              Welch t + permutation  0.0325
    Intervention (circuit)     circuit 1.194184 2.375842e-02              Welch t + permutation  0.0495
     Intervention (limbic)      limbic 1.321615 1.981853e-02              Welch t + permutation  0.0314
Intervention (subcortical) subcortical 1.611886 6.630907e-03              Welch t + permutation  0.0104
[20:19:36] 
[20:19:36] ======================================================================
[20:19:36] v6 COMPLETE
[20:19:36] ==============================================